<a href="https://colab.research.google.com/github/khushikayy/sklearn_library/blob/main/sklearn(7)_Column_Transformer_Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('/content/9 covid_toy.csv')

In [ ]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['has_covid'])
y = df['has_covid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
X_train.shape, y_train.shape, y_test.shape, X_test.shape

((80, 5), (80,), (20,), (20, 5))

# 1 Without Column Transformer

In [ ]:
# missing values fill
from sklearn.impute import SimpleImputer
si = SimpleImputer()

X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [ ]:
# encoding categorical values
from sklearn.preprocessing import OrdinalEncoder
Ordinal = OrdinalEncoder(categories=[['Mild','Strong']])

X_train_cough = Ordinal.fit_transform(X_train[['cough']])
X_test_cough = Ordinal.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [ ]:
# gender and city
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [ ]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape


(80, 1)

In [ ]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough), axis=1)
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough), axis=1)

X_train_transformed.shape

(80, 7)

# 2 With Column Transformer

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
tf = ColumnTransformer(
    transformers=[
        ("tf1",SimpleImputer(),['fever']),
        ("tf2",OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
        ("tf3",OneHotEncoder(drop='first',sparse_output=False),['gender','city'])
    ],
    remainder='passthrough'
)

In [ ]:
tf

ColumnTransformer(remainder='passthrough',
                  transformers=[('tf1', SimpleImputer(), ['fever']),
                                ('tf2',
                                 OrdinalEncoder(categories=[['Mild',
                                                             'Strong']]),
                                 ['cough']),
                                ('tf3',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['gender', 'city'])])

In [ ]:
X_train_tf = tf.fit_transform(X_train)
X_test_tf = tf.transform(X_test)

In [ ]:
X_train_tf

array([[100.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  13.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  65.        ],
       [100.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  11.        ],
       [101.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   1.        ,  19.        ],
       [101.        ,   0.        ,   0.        ,   1.        ,
          0.        ,   0.        ,  64.        ],
       [103.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  60.        ],
       [101.        ,   1.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  14.        ],
       [101.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  15.        ],
       [101.04166667,   1.        ,   0.        ,   0.        ,
          0.    

In [ ]:
df = pd.DataFrame(X_train_tf)
df

,0,1,2,3,4,5,6
0,100.000000,1.0,0.0,0.0,1.0,0.0,13.0
1,98.000000,0.0,0.0,0.0,0.0,1.0,65.0
2,100.000000,0.0,1.0,0.0,0.0,0.0,11.0
3,101.000000,0.0,0.0,0.0,0.0,1.0,19.0
4,101.000000,0.0,0.0,1.0,0.0,0.0,64.0
...,...,...,...,...,...,...,...
75,101.041667,0.0,1.0,1.0,0.0,0.0,38.0
76,104.000000,0.0,1.0,0.0,1.0,0.0,51.0
77,104.000000,1.0,0.0,1.0,0.0,0.0,75.0
78,98.000000,0.0,0.0,0.0,1.0,0.0,31.0
